# NB02 - Opérations Relationnelles sur les Dataframes dans Spark

%md
## A. Chargement des données

Chargement des données dans un dataframe.

Dans ce notebook on utilisera le dataset fournis par Databricks `bakehouse`

In [0]:
from pyspark.sql.functions import *

transactions_df = spark.read.table("samples.bakehouse.sales_transactions")
customers_df = spark.read.table("samples.bakehouse.sales_customers")
franchises_df = spark.read.table("samples.bakehouse.sales_franchises")
suppliers_df = spark.read.table("samples.bakehouse.sales_suppliers")

In [0]:
transactions_df.printSchema()

In [0]:
customers_df.printSchema()

In [0]:
franchises_df.printSchema()

In [0]:
suppliers_df.printSchema()

## B. Jointures basiques

On va enrichir les transactions avec les informations des magasins (franchises)

In [0]:
enriched_transctions = franchises_df.join(
    transactions_df, 
    on = "franchiseID",
    how = "inner"
)

display(enriched_transctions)

La clause `on` utilisée pour la jointure peut contenir une expression : 

In [0]:
enriched_transctions = franchises_df.join(
    transactions_df, 
    on = transactions_df.franchiseID == franchises_df.franchiseID,
    how = "inner"
)

display(enriched_transctions)

Dans le cas précedent toutes les colonnes du Dataset `franchises_df` sont présentes dans le nouveau dataframe.

Une meilleur méthode est de projeter les colonnes que l'on souhaite garder.

In [0]:
enriched_transctions = franchises_df \
    .select(
        "franchiseID",
        col("name").alias("store_name"),
        col("city").alias("store_city"),
        col("country").alias("store_country")
    ) \
    .join(
    transactions_df, 
    on = "franchiseID",
    how = "inner"
    )

display(enriched_transctions)

## C. `Full Outer Join`

Un `full join` (ou `full outer join`) permet de combiner deux tables en conservant toutes les lignes de chaque table, même si elles ne correspondent pas à une clé commune. Les lignes sans correspondance dans l'une ou l'autre table auront des valeurs nulles pour les colonnes de la table opposée. Cela est utile pour obtenir une vue complète de toutes les données, y compris celles qui n'ont pas de correspondance, par exemple pour détecter des valeurs orphelines ou analyser des différences entre deux ensembles de données.

In [0]:
full_join = franchises_df \
  .withColumnRenamed("name", "franchise_name") \
  .join(
    suppliers_df.select("supplierID", col("name").alias("supplier_name")),
    on = "supplierID",
    how = "full_outer"
  )

# on peut vérifier et trouver les lignes qui n'apparaitrait pas dans un inner join
# une ligne ou la franchise ou le supplier serait null
non_matching_records = full_join.filter(
    col("franchiseID").isNull() |
    col("supplier_name").isNull()
  ) \
  .select("franchiseID", "franchise_name", col("supplierID").alias("orphaned_supplier_id"))

display(non_matching_records)

## D. Utilisation de Spark SQL

In [0]:
franchises_df.createOrReplaceTempView("franchises")
suppliers_df.createOrReplaceTempView("suppliers")

In [0]:
%sql
select
  f.franchiseID,
  f.name as franchise_name,
  f.supplierID as orphaned_supplier_id
from franchises f
full outer join suppliers s
on f.supplierID = s.supplierID
where f.franchiseID is null or s.name is null

## D. Opérations sur les ensembles

In [0]:
franchise_suppliers = franchises_df.select("supplierID").distinct()
all_suppliers = suppliers_df.select("supplierID").distinct()

franchises_without_valid_suppliers = franchise_suppliers.subtract(all_suppliers)
display(franchises_without_valid_suppliers)